<a href="https://colab.research.google.com/github/EAName/yelp-reviews/blob/main/Exploratory_Data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# EDA notebook

In [2]:
!git status

fatal: not a git repository (or any of the parent directories): .git


In [3]:
%cd {repository}

[Errno 2] No such file or directory: '{repository}'
/content


# Ingest Data

In [4]:
from google.colab import drive, files
drive.mount('/content/gdrive', force_remount = True)

In [5]:
import pandas as pd
import matplotlib
from matplotlib import pyplot as plt
import seaborn as sns
import statistics
import numpy as np

In [6]:
# First drop yelp_dataset into your drive (Add shortcut to Drive)
business_json_path = '/content/gdrive/MyDrive/yelp_dataset/yelp_academic_dataset_business.json'
df_b = pd.read_json(business_json_path, lines=True)

In [7]:
#keep only restaurant category
df_b = df_b[df_b['categories'].str.contains(
              'restaurant|Restaurant|RESTAURANT',
              case=False, na=False)]

In [8]:
df_b.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
5,CF33F8-E6oudUQ46HnavjQ,Sonic Drive-In,615 S Main St,Ashland City,TN,37015,36.269593,-87.058943,2.0,6,1,"{'BusinessParking': 'None', 'BusinessAcceptsCr...","Burgers, Fast Food, Sandwiches, Food, Ice Crea...","{'Monday': '0:0-0:0', 'Tuesday': '6:0-22:0', '..."
8,k0hlBqXX-Bt0vf1op7Jr1w,Tsevi's Pub And Grill,8025 Mackenzie Rd,Affton,MO,63123,38.565165,-90.321087,3.0,19,0,"{'Caters': 'True', 'Alcohol': 'u'full_bar'', '...","Pubs, Restaurants, Italian, Bars, American (Tr...",None
9,bBDDEgkFA1Otx9Lfe7BZUQ,Sonic Drive-In,2312 Dickerson Pike,Nashville,TN,37207,36.208102,-86.768170,1.5,10,1,"{'RestaurantsAttire': ''casual'', 'Restaurants...","Ice Cream & Frozen Yogurt, Fast Food, Burgers,...","{'Monday': '0:0-0:0', 'Tuesday': '6:0-21:0', '..."
11,eEOYSgkmpB90uNA7lDOMRA,Vietnamese Food Truck,,Tampa Bay,FL,33602,27.955269,-82.456320,4.0,10,1,"{'Alcohol': ''none'', 'OutdoorSeating': 'None'...","Vietnamese, Food, Restaurants, Food Trucks","{'Monday': '11:0-14:0', 'Tuesday': '11:0-14:0'..."


# Elaine's DF Clean of Business


Drop Canada and other states

In [9]:
# look at rows with NC, CO, HI, MT, XMS to see if they are mis-labeled or if they should be dropped

df_b.loc[df_b['state'].isin(["NC", "CO", "HI", "MT", "XMS"])]

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
9101,em474WuTw7-OkhPfX4EohA,IHOP,702 Blowing Rock Rd,Boone,NC,28607,36.378837,-86.463779,2.0,29,0,"{'Alcohol': ''none'', 'RestaurantsTakeOut': 'F...","Breakfast & Brunch, Coffee & Tea, Food, Americ...","{'Monday': '0:0-0:0', 'Tuesday': '7:0-15:0', '..."
9126,1H9WYGFJ4AFwXq7WCOOMtg,Two Elk Restaurant,12000 Skyline Dr,Vail,CO,81657,32.094854,-110.773491,3.5,12,0,"{'GoodForKids': 'True', 'RestaurantsGoodForGro...","Buffets, Restaurants",None
12258,bw90rjFL02A_8rVE34m5Lg,Cosmetic Solutions,1010 Lula Lula Dr,Lula Lula,HI,90261,38.816486,-90.321750,3.5,17,1,"{'BusinessAcceptsCreditCards': 'False', 'Accep...","Cosmetic Surgeons, Pizza, Doctors, Beauty & Sp...",None
48919,gCF0QQWzI9lDouhLfTbYCA,MudMan Food Truck,,Kalispell,MT,59901,27.789678,-82.680746,5.0,6,0,"{'RestaurantsTakeOut': 'True', 'BikeParking': ...","Restaurants, American (Traditional), Food Truc...","{'Tuesday': '17:0-19:30', 'Wednesday': '11:0-1..."
98685,IvQs2hhXKZ4NgZhVZ5jjjQ,Pumpernickles Catering,95a Mount Pleasant,Liverpool,XMS,L3 5TB,38.673049,-90.423156,4.0,5,1,"{'RestaurantsPriceRange2': '2', 'BusinessParki...","Caterers, Breakfast & Brunch, Convenience Stor...",None


Remove these 5 states since they don't belong to a market 

In [10]:
#Drop rows for restaurants in outlier states (4 rows) & Canada

#dfb_restaurants1.loc[dfb_restaurants1['state'].isin(["NC", "CO", "HI", "MT", "XMS"])]

removestates = ["NC", "CO", "HI", "MT", "XMS", "AB"]

df_b = df_b[~ df_b['state'].isin(removestates)]

Total restaurants remaining should be 49,871

Normalize cities and states into DMA markets with market column
For CA cities, I checked that Reno, NV should only contain NV rows and all CA rows belong to Santa Barbara

In [11]:
#Test that CA zipcodes belonging to Reno DMA are not in the df

renoziptest = ["93512", "93517", "93529", "93541", "93546", "95646", "96009", "96056", "96068", "96107", "96109", "96113", "96114", "96117", "96119", "96120", "96121", "96123", "96127", 
               "96128", "96130", "96132", "96133", "96136", "96150", "96151", "96152", "96154", "96155", "96156", "96157", "96158"]

In [12]:
renotestdf = df_b[df_b['postal_code'].isin(renoziptest)]
renotestdf.shape

(0, 14)

In [13]:
#Get list of all states

df_b.state.value_counts()

state
PA    12644
FL     8732
TN     4353
MO     4248
IN     4150
LA     3641
NJ     3343
AZ     2675
NV     1675
ID     1304
CA     1161
IL      983
DE      962
Name: count, dtype: int64

In [14]:
#create market dictionary

market = {'PA': 'Philadelphia', 'DE': 'Philadelphia', 'NJ': 'Philadelphia', 'FL': 'Tampa', 'IN': 'Indianapolis', 'TN': 'Nashville', 'AZ': 'Tucson', 'LA': 'New Orleans', 'MO': 'St. Louis', 'IL': 'St. Louis',
          'NV': 'Reno', 'CA': 'Santa Barbara', 'ID': 'Boise'}

In [15]:
df_b['market']= df_b['state'].map(market)

Remove rows to narrow "Restaurant" Definition to Sit-down restaurants

In [16]:
#Create DF to isolate the rows we want to remove
#Please add more as needed

df_bfast = df_b[df_b['categories'].str.contains('Fast Food|Bakery|Bakeries|Food Trucks|Street Vendors|Coffee| Coffee & Tea|Caterers',
                                                                                 case=False, na=False)]
df_bfast.shape

(12948, 15)

In [17]:
df_b = df_b.drop(df_bfast.index)
df_b.shape

(36923, 15)

Total rows remaining should be 36,923

In [18]:
#Check whether top fast food names are still in the df

fast_list = ["McDonald's", "Subway", "Taco Bell", "Burger King", "Wendy's", "Wawa","Domino's Pizza", "Pizza Hut", "Papa John's Pizza", "Jimmy John's",
             "KFC", "Chick-fil-A", "Popeyes Louisiana Kitchen", "Chipotle Mexican Grill", "Starbucks", "Panera Bread", "Sonic Drive-In", "Arby's",
             "Panda Express", "Waffle House", "IHOP", "Jack in the Box", "Jersey Mike's Subs" , "Steak ’n Shake", "Five Guys", "Dairy Queen" , "QDOBA Mexican Eats", 
             "Little Caesars Pizza", "Hardee's", "Wingstop", "Firehouse Subs", "Dairy Queen Grill & Chill", "Buffalo Wild Wings", "Checkers", "China Wok", 
             "White Castle", "Imo's Pizza", "Papa Murphy's", "PrimoHoagies", "Saladworks", "Marco's Pizza", "QuikTrip", "Philly Pretzel Factory", "Boston Market",
              "Moe's Southwest Grill", "Church's Chicken", "Culver's", "Rally's", "Cracker Barrel Old Country Store", "Tropical Smoothie Café", 
             "Hungry Howie's Pizza & Subs", "McAlister's Deli", "Jet's Pizza", "Hooters", "Little Caesars", "St. Louis Bread", "Friendly's", 
             "Long John Silver's", "Dickey's Barbecue Pit", "Zaxby's Chicken Fingers & Buffalo Wings", "Captain D's", "Chuck E. Cheese", "Carl's Jr",
             "MOD Pizza", "Einstein Bros. Bagels", "Honey Baked Ham Company", "eegee's", "Manhattan Bagel", "Raising Cane's Chicken Fingers", "Noodles & Company", 
             "Smashburger", "Penn Station East Coast Subs", "Tijuana Flats",  "Westshore Pizza", "Zoes Kitchen", "Port of Subs", "Quiznos", 
             "Shake Shack", "Fazoli's"]

In [19]:
print(len(df_b[df_b['name'].isin(fast_list)]))

1891


There are still 1891 rows with the chains in fast_list. We may need to check if we can delete from Pizza and Sandwich category. The only issue with deleting Pizza is that it may delete "Italian" also.

Larger Chains that are still kept within the Total restaurants list to provide comparison for sit-down restaurants:



*   Applebee's Grill + Bar
*   Chili's
* Denny's
* First Watch
* Outback Steakhouse
* China King - there is a chain but there are also independents
* Olive Garden Italian Restaurant
* LongHorn Steakhouse
* Red Lobster
* Ruby Tuesday
* TGI Fridays
* Bob Evans
* Red Robin Gourmet Burgers and Brews
* Beef 'O'Brady's
* Texas Roadhouse
* Carrabba's Italian Grill
* China House
* O'Charley's Restaurant & Bar
* Bonefish Grill
* New China





